# FuelGuard Reinforcement Learning Walkthrough

This notebook demonstrates:
1. Gymnasium FuelSupplyEnv initialization wrapping BUP Fuel Supply Simulator
2. State space representation and observation extraction
3. Action space formulation and deterministic guardrail validation
4. Stepping the environment with multi-objective rewards
5. PPO Agent inference and integration with RAG policy retrieval

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("../../backend").resolve()))

from app.rl.environment.fuel_env import FuelSupplyEnv
from app.rl.environment.action import RLActionSpace, validate_action
from app.rl.inference.predictor import RLPredictor
from app.rag.pipeline import get_rag_pipeline
print("FuelGuard RL and RAG modules successfully imported.")

In [ ]:
# Initialize Environment
env = FuelSupplyEnv(max_ticks=96)
obs, info = env.reset(seed=42)
print(f"Observation shape: {obs.shape}")
print(f"Action space size: {env.action_space.n}")

In [ ]:
# Step the environment with a candidate action
action = 1  # candidate dispatch action
next_obs, reward, terminated, truncated, step_info = env.step(action)
print(f"Step Reward: {reward:.3f}")
print(f"Reward Breakdown: {step_info[reward_breakdown]}")
print(f"Action Valid: {step_info[action_valid]}")

In [ ]:
# Production Predictor & Guardrail Check
predictor = RLPredictor()
rec = predictor.predict_recommendation(env.current_snapshot)
print("RL Recommendation:")
import json
print(json.dumps(rec, indent=2))

In [ ]:
# Query RAG Knowledge Base for Applicable Policies
rag = get_rag_pipeline()
query = "What policy applies to minimum depot reserve?"
search_res = rag.search(query, category="rules_policies", top_k=2)
for r in search_res:
    print(f"[SOURCE: {r.source}] Score: {r.score:.3f}")
    print(r.content[:200] + "...")